# Driver Intent — One-Shot Training Notebook

Two runs: vanilla baseline + full weak-supervision pipeline. Total ~3.5 hrs on a free T4.

**Before running:** make sure these exist on Drive:
- `MyDrive/intent_data/hdd_train.json`
- `MyDrive/intent_data/hdd_cleaned_validated.json`
- (output dir, will be auto-created) `MyDrive/intent_models/`

Run cells top to bottom. Sit on the tab. Don't let it idle past 90 min.

## 1. Setup — mount Drive, clone repo, install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf driver_intent_monitoring_system
!git clone https://github.com/AranD3V/driver_intent_monitoring_system.git
%cd driver_intent_monitoring_system

In [ ]:
# Install only the deps the training path needs (skip MediaPipe/CARLA/etc).
!pip install -q -r requirements-colab.txt

In [ ]:
# Link Drive folders so checkpoints survive disconnect.
!mkdir -p /content/drive/MyDrive/intent_data /content/drive/MyDrive/intent_models /content/drive/MyDrive/intent_reports
!rm -rf data models reports
!ln -s /content/drive/MyDrive/intent_data    data
!ln -s /content/drive/MyDrive/intent_models  models
!ln -s /content/drive/MyDrive/intent_reports reports
!ls -la data/ | head

In [ ]:
# Sanity check: GPU + data files exist.
import torch, json, os
print('CUDA available :', torch.cuda.is_available())
print('GPU            :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
for f in ['data/hdd_train.json', 'data/hdd_cleaned_validated.json']:
    if os.path.exists(f):
        d = json.load(open(f))
        print(f'{f}: {len(d)} sequences')
    else:
        print(f'MISSING: {f}')

## 2. Run A — vanilla baseline

5-fold CV on the original `hdd_train.json`, no augmentation, no sample weights. ~90 min.

In [ ]:
!python scripts/train_intent.py train-kfold \
    --data data/hdd_train.json \
    --output models/baseline.pth \
    --epochs 40 --batch 32 --seq-len 50 --stride 10 \
    --hidden 128 --dropout 0.3 --lr 3e-4 \
    --early-stop 12 --seed 42 --overwrite

## 3. Run B — full weak-supervision pipeline

5-fold CV on `hdd_cleaned_validated.json` with `--weak-aug --use-weak-weights`. ~90 min.

In [ ]:
!python scripts/train_intent.py train-kfold \
    --data data/hdd_cleaned_validated.json \
    --output models/weak.pth \
    --epochs 40 --batch 32 --seq-len 50 --stride 10 \
    --hidden 128 --dropout 0.3 --lr 3e-4 \
    --early-stop 12 --seed 42 --overwrite \
    --weak-aug --use-weak-weights

## 4. Evaluate both — same eval set, fair comparison

In [ ]:
# Eval both on the original hdd_train.json (vanilla labels) so the comparison is honest:
# both models are scored against the *same* ground truth.
!python scripts/train_intent.py evaluate \
    --output models/baseline.pth --data data/hdd_train.json \
    --batch 32 --seq-len 50 --otc | tee reports/eval_baseline.txt

In [ ]:
!python scripts/train_intent.py evaluate \
    --output models/weak.pth --data data/hdd_train.json \
    --batch 32 --seq-len 50 --otc | tee reports/eval_weak.txt

## 5. Save canonical checkpoint

Whichever scored higher becomes `intent_canonical.pth`.

In [ ]:
import re, shutil, glob

def parse_acc(path):
    txt = open(path).read()
    m = re.search(r'accuracy\s+([\d.]+)', txt) or re.search(r'(\d{2,3}\.\d)%', txt)
    return float(m.group(1)) if m else -1.0

acc_b = parse_acc('reports/eval_baseline.txt')
acc_w = parse_acc('reports/eval_weak.txt')
print(f'baseline acc = {acc_b}')
print(f'weak    acc = {acc_w}')

winner = 'models/weak.pth' if acc_w >= acc_b else 'models/baseline.pth'
shutil.copy(winner, 'models/intent_canonical.pth')
print(f'\nCanonical -> {winner}')
print('All checkpoints in Drive/intent_models:', sorted(glob.glob('models/*.pth')))

In [ ]:
# Summary table for the writeup.
print('='*60)
print(f'{"Run":<25}{"Eval acc":>15}{"Delta":>15}')
print('='*60)
print(f'{"baseline":<25}{acc_b:>15.2f}{" ":>15}')
print(f'{"+ weak pipeline":<25}{acc_w:>15.2f}{(acc_w - acc_b):>+15.2f}')
print('='*60)